# Model Optimization Across Transformer Architectures
### Cross-domain comparison: BERT (NLP) vs. ViT (Vision)

**Research question:** How do model optimization techniques (quantization, pruning,
knowledge distillation, low-rank factorization) behave across Transformer
architectures in NLP and computer vision, and does the best optimization
strategy depend on the architecture and domain?

This notebook is the experiment driver / narrative layer. Heavy lifting lives in
`src/` and `scripts/` so it stays reusable outside the notebook; this notebook
imports those modules and focuses on results, tables, and figures.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

from src.evaluation.model_stats import record_environment, set_all_seeds
set_all_seeds(42)
print(record_environment())

## 1. Environment & Reproducibility
Fixed seed (42) across `random`/`numpy`/`torch`. Exact library versions and
hardware are captured by `record_environment()` above and logged with every
experiment via `src/evaluation/model_stats.log_experiment`.

## 2. Shared Evaluation Utilities
All techniques are scored with the *same* functions (`src/evaluation/metrics.py`,
`src/evaluation/benchmark.py`) so before/after and cross-technique comparisons are
apples-to-apples:
- Task performance: accuracy + macro-F1
- Model size: real serialized checkpoint size (MB)
- Parameter count: total vs. non-zero/effective
- Sparsity %, compression ratio, performance retention %
- Latency: warmed-up, repeated-run mean/std/p50/p95, throughput (samples/sec)

## 3. NLP Branch — BERT-base-uncased on AG News

In [ ]:
from src.data.nlp_data import load_agnews_tokenized
nlp_data = load_agnews_tokenized()
print(f"train={len(nlp_data.train)} val={len(nlp_data.val)} test={len(nlp_data.test)}")

### 3.1 Baseline
Fine-tuning is run via `scripts/train_bert.py` (kept as a script, not inline, so it
can run unattended / be resumed). Loads the saved checkpoint here for evaluation.

```bash
python scripts/train_bert.py --epochs 3 --batch_size 16
```

In [ ]:
from transformers import AutoModelForSequenceClassification
from torch.utils.data import DataLoader
from scripts.run_experiments import score

bert_baseline = AutoModelForSequenceClassification.from_pretrained(
    'results/checkpoints/bert_baseline'
).to('cuda' if torch.cuda.is_available() else 'cpu')

test_loader = DataLoader(
    nlp_data.test, batch_size=16,
    collate_fn=lambda b: {
        'input_ids': torch.stack([x['input_ids'] for x in b]),
        'attention_mask': torch.stack([x['attention_mask'] for x in b]),
        'labels': torch.stack([x['label'] for x in b]),
    }
)
bert_baseline_metrics = score(bert_baseline, test_loader, ('input_ids', 'attention_mask'))
bert_baseline_metrics

### 3.2 Quantization, Pruning, Distillation, Bonus Low-Rank
Run once via the orchestration script (keeps this notebook fast to re-render):
```bash
python scripts/run_experiments.py --model bert
```
Results are appended to `results/metrics.csv`; loaded and visualized below.

## 4. Vision Branch — ViT-Base-Patch16-224 on CIFAR-10

In [ ]:
from src.data.vision_data import load_cifar10_processed
vision_data = load_cifar10_processed()
print(f"train={len(vision_data.train)} val={len(vision_data.val)} test={len(vision_data.test)}")

### 4.1 Baseline
```bash
python scripts/train_vit.py --epochs 5 --batch_size 32
```
### 4.2 Quantization, Pruning, Distillation
```bash
python scripts/run_experiments.py --model vit
```

## 5. Optional Combined Optimization (Pruning + Quantization)
Run as part of `run_experiments.py` for both architectures (Priority 7). Kept
clearly separate from the independent single-technique results in the table
below, per Section 10's requirement to distinguish independent vs. combined runs.

## 6. Cross-Architecture Results

In [ ]:
results = pd.read_csv('../results/metrics.csv')
results[['experiment', 'accuracy', 'macro_f1', 'size_mb', 'sparsity_pct',
         'compression_ratio', 'performance_retention_pct']]

## 7. Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

sns.barplot(data=results, x='experiment', y='accuracy', ax=axes[0, 0])
axes[0, 0].set_title('Accuracy by experiment'); axes[0, 0].tick_params(axis='x', rotation=75)

sns.barplot(data=results, x='experiment', y='size_mb', ax=axes[0, 1])
axes[0, 1].set_title('Model size (MB) by experiment'); axes[0, 1].tick_params(axis='x', rotation=75)

sns.barplot(data=results, x='experiment', y='sparsity_pct', ax=axes[1, 0])
axes[1, 0].set_title('Sparsity % by experiment'); axes[1, 0].tick_params(axis='x', rotation=75)

sns.scatterplot(data=results, x='size_mb', y='accuracy', hue='experiment', s=100, ax=axes[1, 1])
axes[1, 1].set_title('Accuracy vs. model size (Pareto view)')

plt.tight_layout()
plt.savefig('../results/figures/cross_architecture_summary.png', dpi=150)
plt.show()

## 8. Discussion, Conclusions, Limitations

**Discussion.** *(Fill in after `run_experiments.py` produces `results/metrics.csv`
for both architectures — compare best accuracy-efficiency trade-off per
architecture, best compression technique, best practical latency improvement,
and whether the best strategy is architecture/domain-dependent.)*

**Limitations.** Sparsity-based pruning here is unstructured (mask-based); real
wall-clock speedups on commodity hardware require the structured-pruning +
compaction path noted in `src/optimizations/pruning.py`. Dynamic INT8
quantization is CPU-only via PyTorch's native backend — GPU-accelerated INT4 is
documented but not exercised (`quantize_bitsandbytes_4bit`). Distillation uses a
fixed 4-layer student architecture as a first pass; a small student
architecture search was out of scope given the project timeline.

**Conclusions.** *(Summarize the strongest technique per architecture and any
domain-dependent pattern once results are populated.)*